# 05b - Feature Engineering France : silver_france → gold_france

**Architecture Medallion – Couche Gold France métropolitaine**

Ce notebook construit la table ML-ready `gold_france.features_communes` en :
1. Agrégeant les résultats des **présidentielles T1** par commune × année (2002–2022)
2. Joignant les features socio-économiques depuis `silver_france.*`
3. Ajoutant la feature `typologie_territoire` (urbain / periurbain / rural)
4. Appliquant l'imputation médiane départementale (puis nationale) pour les valeurs manquantes

### Scrutin de référence
Présidentielles T1 : 2002, 2007, 2012, 2017, 2022 (~34 000 communes × 5 ans = ~170 000 lignes)

### Typologie territoire
Calculée depuis la **population totale RP 2022** (somme des tranches d'âge dans le fichier diplomes) :
- `urbain`    : population >= 10 000 hab (seuil UNESCO/INSEE urbain)
- `periurbain`: population >= 2 000 hab
- `rural`     : population < 2 000 hab

## 0. Configuration & imports

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Configuration inline
PG_HOST     = os.environ.get("POSTGRES_HOST",     "postgres")
PG_PORT     = os.environ.get("POSTGRES_PORT",     "5432")
PG_DB       = os.environ.get("POSTGRES_DB",       "mspr813")
PG_USER     = os.environ.get("POSTGRES_USER",     "mspr_user")
PG_PASSWORD = os.environ.get("POSTGRES_PASSWORD", "mspr_password")

SCHEMA_IN  = 'silver_france'
SCHEMA_OUT = 'gold_france'

BRONZE_DIR = '/app/data/bronze'

DB_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
engine = create_engine(DB_URL, pool_pre_ping=True)

with engine.connect() as conn:
    print("PostgreSQL OK :", conn.execute(text("SELECT current_database()")).scalar())

# Vérification des sources
with engine.connect() as conn:
    for tbl in ['elections','naissances','deces','revenus','csp','diplomes','referentiel_communes']:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {SCHEMA_IN}.{tbl}")).scalar()
        print(f"  {SCHEMA_IN}.{tbl} : {n:,} lignes")

PostgreSQL OK : mspr813


  silver_france.elections : 25,996,190 lignes
  silver_france.naissances : 591,328 lignes
  silver_france.deces : 591,328 lignes
  silver_france.revenus : 34,844 lignes
  silver_france.csp : 34,825 lignes
  silver_france.diplomes : 34,738 lignes
  silver_france.referentiel_communes : 34,851 lignes


## 1. Mapping nuances → blocs politiques

In [2]:
# Mapping nuance_liste -> bloc politique
NUANCE_TO_BLOC = {
    # ---- GAUCHE ----
    'EXG': 'Gauche', 'LEXG': 'Gauche', 'LXG': 'Gauche', 'DXG': 'Gauche',
    'COM': 'Gauche', 'LCOM': 'Gauche',
    'SOC': 'Gauche', 'LSOC': 'Gauche',
    'DVG': 'Gauche', 'LDVG': 'Gauche',
    'FG':  'Gauche', 'LFG':  'Gauche',
    'NUP': 'Gauche',
    'LFI': 'Gauche', 'FI': 'Gauche',
    'ECO': 'Gauche', 'LECO': 'Gauche',
    'VEC': 'Gauche', 'LVEC': 'Gauche',
    'LCR': 'Gauche', 'LO': 'Gauche',
    'PG':  'Gauche', 'LPG': 'Gauche',
    'BC-EXG': 'Gauche', 'BC-DVG': 'Gauche', 'BC-FG': 'Gauche',
    'BC-SOC': 'Gauche', 'BC-FI': 'Gauche', 'BC-COM': 'Gauche',
    'BC-VEC': 'Gauche', 'BC-ECO': 'Gauche', 'BC-PG': 'Gauche',
    'BC-UCG': 'Gauche',
    'GAU': 'Gauche', 'PREP': 'Gauche',
    'MELE': 'Gauche', 'HOLL': 'Gauche', 'ROYA': 'Gauche',
    'BUFF': 'Gauche', 'VOYN': 'Gauche', 'BESA': 'Gauche',
    'ARTH': 'Gauche', 'POUT': 'Gauche', 'GLUC': 'Gauche',
    'HUE': 'Gauche', 'JOSP': 'Gauche', 'BOVE': 'Gauche',
    'TAUB': 'Gauche', 'JOLY': 'Gauche', 'MAME': 'Gauche',
    # ---- CENTRE ----
    'ENS': 'Centre', 'REM': 'Centre', 'LREM': 'Centre',
    'BC-REM': 'Centre', 'BC-MDM': 'Centre',
    'BAYR': 'Centre', 'LCOP': 'Centre',
    'UDF': 'Centre', 'LUDF': 'Centre',
    'MODM': 'Centre', 'LMDM': 'Centre', 'MDM': 'Centre', 'MDC': 'Centre',
    'UDI': 'Centre', 'LUDI': 'Centre', 'BC-UDI': 'Centre',
    'LUD': 'Centre',
    'UG':  'Centre', 'LUG': 'Centre', 'LUC': 'Centre', 'LUGE': 'Centre',
    'DVC': 'Centre', 'LDVC': 'Centre', 'LCMD': 'Centre',
    'BC-UC': 'Centre', 'BC-UCD': 'Centre',
    'NCE': 'Centre', 'M': 'Centre', 'M-NC': 'Centre',
    'BC-DVC': 'Centre',
    'CHEV': 'Centre', 'LEPAGE': 'Centre', 'LASA': 'Centre',
    # ---- DROITE ----
    'UMP': 'Droite', 'LUMP': 'Droite', 'BC-UMP': 'Droite',
    'LR': 'Droite', 'LLR': 'Droite', 'BC-LR': 'Droite',
    'RPR': 'Droite', 'RPF': 'Droite',
    'DVD': 'Droite', 'LDVD': 'Droite', 'BC-DVD': 'Droite',
    'DLF': 'Droite', 'LDLF': 'Droite', 'BC-DLF': 'Droite',
    'MPF': 'Droite', 'UDFD': 'Droite',
    'SARK': 'Droite', 'CHIR': 'Droite', 'MADE': 'Droite',
    'PECA': 'Droite', 'FILL': 'Droite',
    'VILL': 'Droite', 'NIHO': 'Droite', 'SCHI': 'Droite',
    'BOUT': 'Droite', 'DUPO': 'Droite', 'CHEM': 'Droite',
    'SAIN': 'Droite', 'LAGU': 'Droite', 'CPNT': 'Droite',
    # ---- EXTREME DROITE ----
    'FN': 'ExtremeDroite', 'LFN': 'ExtremeDroite',
    'RN': 'ExtremeDroite', 'LRN': 'ExtremeDroite', 'BC-RN': 'ExtremeDroite',
    'MNR': 'ExtremeDroite', 'MEGR': 'ExtremeDroite',
    'LEPA': 'ExtremeDroite',
    'EXD': 'ExtremeDroite', 'LEXD': 'ExtremeDroite', 'LXD': 'ExtremeDroite',
    'BC-EXD': 'ExtremeDroite', 'BC-FN': 'ExtremeDroite',
    'REC': 'ExtremeDroite', 'LREC': 'ExtremeDroite',
    'DTE': 'ExtremeDroite', 'FRN': 'ExtremeDroite', 'MNA': 'ExtremeDroite',
    'UXD': 'ExtremeDroite',
    # ---- DIVERS ----
    'DIV': 'Divers', 'LDIV': 'Divers', 'LDV': 'Divers',
    'AUT': 'Divers', 'LAUT': 'Divers',
    'BC-DIV': 'Divers', 'REG': 'Divers',
    'LDD': 'Divers', 'DSV': 'Divers',
    'BC-RDG': 'Divers', 'RDG': 'Divers', 'PRG': 'Divers', 'PRV': 'Divers',
    'ALLI': 'Divers', 'LAGE': 'Divers', 'LDR': 'Divers',
}

# Mapping par nom de candidat (présidentielles, quand nuance absente)
CANDIDAT_PRES_TO_BLOC = {
    'ARTHAUD': 'Gauche', 'POUTOU': 'Gauche', 'MELENCHON': 'Gauche',
    'MÉLENCHON': 'Gauche', 'JADOT': 'Gauche', 'HIDALGO': 'Gauche',
    'ROUSSEL': 'Gauche', 'HAMON': 'Gauche', 'HOLLANDE': 'Gauche',
    'ROYAL': 'Gauche', 'JOSPIN': 'Gauche', 'BUFFET': 'Gauche',
    'VOYNET': 'Gauche', 'BESANCENOT': 'Gauche', 'GLUCKSTEIN': 'Gauche',
    'HUE': 'Gauche', 'MAMERE': 'Gauche', 'BOVÉ': 'Gauche',
    'TAUBIRA': 'Gauche', 'JOLY': 'Gauche',
    'MACRON': 'Centre', 'BAYROU': 'Centre', 'CHEVENEMENT': 'Centre',
    'LEPAGE': 'Centre', 'LASSALLE': 'Centre',
    'SARKOZY': 'Droite', 'CHIRAC': 'Droite', 'FILLON': 'Droite',
    'PÉCRESSE': 'Droite', 'MADELIN': 'Droite', 'de VILLIERS': 'Droite',
    'NIHOUS': 'Droite', 'SCHIVARDI': 'Droite', 'BOUTIN': 'Droite',
    'DUPONT-AIGNAN': 'Droite', 'SAINT-JOSSE': 'Droite', 'LAGUILLER': 'Droite',
    'ASSELINEAU': 'Droite', 'CHEMINADE': 'Divers',
    'LE PEN': 'ExtremeDroite', 'ZEMMOUR': 'ExtremeDroite',
    'MEGRET': 'ExtremeDroite',
}

print(f"Mapping nuances : {len(NUANCE_TO_BLOC)} entrées")
print(f"Mapping candidats présidentiels : {len(CANDIDAT_PRES_TO_BLOC)} entrées")

Mapping nuances : 148 entrées
Mapping candidats présidentiels : 42 entrées


## 2. Agrégation électorale présidentielles T1

Traitement par chunks pour gérer les ~4M lignes présidentielles France entière.

In [3]:
import time

def get_bloc(nuance, candidat):
    """Retourne le bloc politique d'une ligne électorale."""
    # Priorité 1 : nuance_liste
    if pd.notna(nuance) and nuance != '':
        b = NUANCE_TO_BLOC.get(str(nuance).strip())
        if b:
            return b
    # Priorité 2 : nom_candidat (présidentielles)
    if pd.notna(candidat) and candidat != '':
        nom = str(candidat).upper().strip()
        b = CANDIDAT_PRES_TO_BLOC.get(nom)
        if b:
            return b
        # Recherche partielle
        for k, v in CANDIDAT_PRES_TO_BLOC.items():
            if k.upper() in nom or nom in k.upper():
                return v
    return 'Divers'

t0 = time.time()

# Charger présidentielles T1 depuis silver_france
print("Chargement présidentielles T1 depuis silver_france...")
df_elec = pd.read_sql("""
    SELECT code_commune, annee, nuance_liste, nom_candidat, nb_voix
    FROM silver_france.elections
    WHERE type_election = 'pres' AND tour = 1
      AND nb_voix IS NOT NULL
""", engine)

print(f"  {len(df_elec):,} lignes chargées en {time.time()-t0:.1f}s")
print(f"  Années : {sorted(df_elec['annee'].unique())}")
print(f"  Communes : {df_elec['code_commune'].nunique():,}")

Chargement présidentielles T1 depuis silver_france...


  3,884,058 lignes chargées en 7.5s
  Années : [2002, 2007, 2012, 2017, 2022]
  Communes : 34,800


In [4]:
t1 = time.time()

# Attribution des blocs (vectorisée)
print("Attribution des blocs...")
df_elec['nuance_liste'] = df_elec['nuance_liste'].fillna('')
df_elec['nom_candidat'] = df_elec['nom_candidat'].fillna('')

# Mapping vectorisé
df_elec['bloc'] = df_elec['nuance_liste'].map(NUANCE_TO_BLOC)

# Pour les lignes sans bloc via nuance, tenter via candidat
mask_no_bloc = df_elec['bloc'].isna()
df_elec.loc[mask_no_bloc, 'bloc'] = (
    df_elec.loc[mask_no_bloc, 'nom_candidat']
    .str.upper().str.strip()
    .map(CANDIDAT_PRES_TO_BLOC)
)

# Fallback Divers
df_elec['bloc'] = df_elec['bloc'].fillna('Divers')

print(f"  Attribution terminée en {time.time()-t1:.1f}s")
print("\nDistribution blocs (nb_voix) :")
print(df_elec.groupby('bloc')['nb_voix'].sum().sort_values(ascending=False).to_string())

Attribution des blocs...


  Attribution terminée en 0.9s

Distribution blocs (nb_voix) :
bloc
Gauche           55562008
Droite           42998515
ExtremeDroite    33556715
Centre           32102317
Divers              61879


In [5]:
t2 = time.time()

# Agréger par commune × année × bloc
print("Agrégation par commune × année × bloc...")
df_bloc = (
    df_elec
    .groupby(['code_commune', 'annee', 'bloc'], as_index=False)['nb_voix']
    .sum()
)

# Total voix par commune × année
df_total = (
    df_elec
    .groupby(['code_commune', 'annee'], as_index=False)['nb_voix']
    .sum()
    .rename(columns={'nb_voix': 'total_voix'})
)

df_bloc = df_bloc.merge(df_total, on=['code_commune', 'annee'])
df_bloc['pct'] = df_bloc['nb_voix'] / df_bloc['total_voix'] * 100

# Pivot : une ligne par commune × année
df_pivot = df_bloc.pivot_table(
    index=['code_commune', 'annee'],
    columns='bloc',
    values='pct',
    fill_value=0
).reset_index()
df_pivot.columns.name = None

# Garantir toutes les colonnes bloc
for bloc in ['Gauche', 'Centre', 'Droite', 'ExtremeDroite', 'Divers']:
    if bloc not in df_pivot.columns:
        df_pivot[bloc] = 0.0

df_pivot = df_pivot.rename(columns={
    'Gauche':        'pct_gauche',
    'Centre':        'pct_centre',
    'Droite':        'pct_droite',
    'ExtremeDroite': 'pct_extremedroite',
    'Divers':        'pct_divers',
})

# Bloc dominant
blocs_cols  = ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']
bloc_labels = ['Gauche', 'Centre', 'Droite', 'ExtremeDroite']
df_pivot['bloc_dominant'] = df_pivot[blocs_cols].idxmax(axis=1).map(
    dict(zip(blocs_cols, bloc_labels))
)

print(f"  Pivot terminé en {time.time()-t2:.1f}s")
print(f"  Shape pivot : {df_pivot.shape}")
print(f"  Communes : {df_pivot['code_commune'].nunique():,}")
print("\nDistribution blocs dominants par année :")
print(df_pivot.groupby(['annee','bloc_dominant']).size().unstack(fill_value=0).to_string())

Agrégation par commune × année × bloc...


  Pivot terminé en 0.5s
  Shape pivot : (173925, 8)
  Communes : 34,800

Distribution blocs dominants par année :
bloc_dominant  Centre  Droite  ExtremeDroite  Gauche
annee                                               
2002               62   24692           2691    7332
2007              484   22211             92   11993
2012               30    8931           1594   24239
2017             4019   10425          11817    8530
2022             9485      26          20193    5079


## 3. Features socio-économiques depuis silver_france

In [6]:
print("Chargement features socio-économiques...")

# Référentiel communes
df_ref = pd.read_sql("""
    SELECT code_insee AS code_commune, libelle, code_dep
    FROM silver_france.referentiel_communes
""", engine)
print(f"  Référentiel : {len(df_ref):,} communes")

# Naissances
df_nais = pd.read_sql("""
    SELECT code_commune, annee, nb_naissances
    FROM silver_france.naissances
""", engine)
print(f"  Naissances : {len(df_nais):,} lignes, années {sorted(df_nais['annee'].unique()[:3])}...")

# Décès
df_deces = pd.read_sql("""
    SELECT code_commune, annee, nb_deces
    FROM silver_france.deces
""", engine)
print(f"  Décès : {len(df_deces):,} lignes")

# Revenus (2022)
df_rev = pd.read_sql("""
    SELECT code_commune, annee, mediane_revenu_disp, gini, taux_pauvrete
    FROM silver_france.revenus
""", engine)
print(f"  Revenus : {len(df_rev):,} lignes")

# CSP (2022)
df_csp = pd.read_sql("""
    SELECT code_commune,
           cadres_emploi, ouvriers_emploi, employes_emploi, artisans_emploi,
           prof_interm_emploi, agriculteurs_emploi,
           COALESCE(cadres_emploi,0) + COALESCE(ouvriers_emploi,0) +
           COALESCE(employes_emploi,0) + COALESCE(artisans_emploi,0) +
           COALESCE(prof_interm_emploi,0) + COALESCE(agriculteurs_emploi,0) AS total_emploi
    FROM silver_france.csp
    WHERE annee = 2022
""", engine)

for col, new_col in [('cadres_emploi','cadres_pct'),('ouvriers_emploi','ouvriers_pct'),
                      ('employes_emploi','employes_pct'),('artisans_emploi','artisans_pct')]:
    df_csp[new_col] = np.where(
        df_csp['total_emploi'] > 0,
        (df_csp[col] / df_csp['total_emploi'] * 100).round(3),
        np.nan
    )
df_csp = df_csp[['code_commune','cadres_pct','ouvriers_pct','employes_pct','artisans_pct']]
print(f"  CSP : {len(df_csp):,} communes")

# Diplômes (2022)
df_dipl = pd.read_sql("""
    SELECT code_commune, nscol15p,
           COALESCE(sup_bac2,0) + COALESCE(sup_bac34,0) + COALESCE(sup_bac5,0) AS bac_plus,
           COALESCE(sans_diplome,0) AS sans_diplome
    FROM silver_france.diplomes
    WHERE annee = 2022
""", engine)

df_dipl['pct_bac_plus']    = np.where(
    df_dipl['nscol15p'] > 0,
    (df_dipl['bac_plus']     / df_dipl['nscol15p'] * 100).round(3),
    np.nan
)
df_dipl['pct_sans_diplome'] = np.where(
    df_dipl['nscol15p'] > 0,
    (df_dipl['sans_diplome'] / df_dipl['nscol15p'] * 100).round(3),
    np.nan
)
df_dipl = df_dipl[['code_commune','pct_bac_plus','pct_sans_diplome']]
print(f"  Diplômes : {len(df_dipl):,} communes")

Chargement features socio-économiques...
  Référentiel : 34,851 communes


  Naissances : 591,328 lignes, années [2015, 2018, 2021]...


  Décès : 591,328 lignes
  Revenus : 34,844 lignes
  CSP : 34,825 communes


  Diplômes : 34,738 communes


## 4. Typologie territoire

Calculée depuis la **population totale RP 2022** disponible dans le fichier diplomes Bronze.

Seuils (population communale, approche INSEE) :
- `urbain`     : >= 10 000 hab
- `periurbain` : >= 2 000 hab
- `rural`      : < 2 000 hab

In [7]:
print("Calcul de la typologie territoire depuis le fichier Bronze diplomes...")

dipl_path = os.path.join(BRONZE_DIR, 'diplomes_formation_2022', 'base-cc-diplomes-formation-2022.xlsx')

df_pop_raw = pd.read_excel(
    dipl_path,
    sheet_name='COM_2022',
    skiprows=5,
    dtype=str
)

# Population = somme des tranches d'âge disponibles (2 ans et plus)
pop_cols = [c for c in df_pop_raw.columns if c.startswith('P22_POP')]
print(f"  Colonnes population : {pop_cols}")

df_pop = df_pop_raw[['CODGEO'] + pop_cols].copy()
df_pop['code_commune'] = df_pop['CODGEO'].astype(str).str.zfill(5)

# Filtrer France métropolitaine (01-95 + 2A/2B)
mask_metro = (
    (df_pop['code_commune'].str[:2].between('01', '95')) |
    (df_pop['code_commune'].str[:2].isin(['2A', '2B']))
)
# Exclure DOM-TOM (971-976)
mask_dom = df_pop['code_commune'].str[:2].isin(['97', '98', '99'])
# Exclure code_dep '20' (Corse fusionnée, garder 2A/2B)
mask_corse_old = df_pop['code_commune'].str[:2] == '20'

df_pop = df_pop[mask_metro & ~mask_dom].copy()

for col in pop_cols:
    df_pop[col] = pd.to_numeric(df_pop[col], errors='coerce')

df_pop['population_totale'] = df_pop[pop_cols].sum(axis=1, skipna=True)

# Estimation : les tranches ne couvrent que 2 ans et plus → majorer de ~3% (enfants 0-2 ans)
# On garde tel quel pour les seuils relatifs

# Appliquer la typologie
def assign_typologie(pop):
    if pd.isna(pop) or pop == 0:
        return 'rural'
    elif pop >= 10000:
        return 'urbain'
    elif pop >= 2000:
        return 'periurbain'
    else:
        return 'rural'

df_pop['typologie_territoire'] = df_pop['population_totale'].apply(assign_typologie)
df_pop = df_pop[['code_commune', 'population_totale', 'typologie_territoire']]

print(f"  Population calculée : {len(df_pop):,} communes")
print("\nDistribution typologies :")
print(df_pop['typologie_territoire'].value_counts().to_string())

Calcul de la typologie territoire depuis le fichier Bronze diplomes...


  Colonnes population : ['P22_POP0205', 'P22_POP0610', 'P22_POP1114', 'P22_POP1517', 'P22_POP1824', 'P22_POP2529', 'P22_POP30P']
  Population calculée : 34,746 communes

Distribution typologies :
typologie_territoire
rural         29482
periurbain     4311
urbain          953


## 5. Construction du Gold DataFrame

In [8]:
print("Construction du Gold DataFrame...")

# Base : pivot élections
df_gold = df_pivot.copy()

# Joindre référentiel
df_gold = df_gold.merge(df_ref, on='code_commune', how='left')

# Préparer nais/deces fusionnés
df_nais_deces = df_nais.merge(df_deces, on=['code_commune','annee'], how='outer')
df_nais_deces['solde_naturel'] = (
    df_nais_deces['nb_naissances'].fillna(0) - df_nais_deces['nb_deces'].fillna(0)
).astype('Int64')

# Fonction de jointure sur l'année la plus proche
def join_closest_year_df(df_main, df_feat, feat_cols):
    """Jointure sur l'année la plus proche disponible dans df_feat."""
    feat_years = sorted(df_feat['annee'].unique())
    
    # Créer le mapping année principale -> année feat la plus proche
    main_years = df_main['annee'].unique()
    year_map = {y: min(feat_years, key=lambda fy: abs(fy - y)) for y in main_years}
    
    # Ajouter colonne annee_feat
    df_main = df_main.copy()
    df_main['_annee_feat'] = df_main['annee'].map(year_map)
    
    df_feat_renamed = df_feat.copy()
    df_feat_renamed = df_feat_renamed.rename(columns={'annee': '_annee_feat'})
    
    df_merged = df_main.merge(
        df_feat_renamed[['code_commune', '_annee_feat'] + feat_cols],
        on=['code_commune', '_annee_feat'],
        how='left'
    ).drop(columns=['_annee_feat'])
    
    return df_merged

# Joindre naissances/décès
df_gold = join_closest_year_df(
    df_gold, df_nais_deces,
    ['nb_naissances', 'nb_deces', 'solde_naturel']
)

# Joindre revenus
if len(df_rev) > 0:
    df_gold = join_closest_year_df(
        df_gold, df_rev,
        ['mediane_revenu_disp', 'gini', 'taux_pauvrete']
    )
else:
    print("WARN: revenus vide")
    for c in ['mediane_revenu_disp', 'gini', 'taux_pauvrete']:
        df_gold[c] = np.nan

# Joindre CSP et Diplômes (statiques 2022)
df_gold = df_gold.merge(df_csp,  on='code_commune', how='left')
df_gold = df_gold.merge(df_dipl, on='code_commune', how='left')

# Joindre typologie territoire
df_gold = df_gold.merge(
    df_pop[['code_commune', 'typologie_territoire']],
    on='code_commune',
    how='left'
)
df_gold['typologie_territoire'] = df_gold['typologie_territoire'].fillna('rural')

print(f"Gold shape avant imputation : {df_gold.shape}")

# Taux de remplissage
fill_rate = (df_gold.notna().sum() / len(df_gold) * 100).round(1)
print("\nTaux de remplissage (%) :")
print(fill_rate[fill_rate < 100].to_string())

Construction du Gold DataFrame...


Gold shape avant imputation : (173925, 23)

Taux de remplissage (%) :
nb_naissances          99.8
nb_deces               99.8
solde_naturel          99.8
mediane_revenu_disp    89.7
gini                   15.1
taux_pauvrete           0.0
cadres_pct             99.4
ouvriers_pct           99.4
employes_pct           99.4
artisans_pct           99.4
pct_bac_plus           99.8
pct_sans_diplome       99.8


## 6. Imputation des valeurs manquantes

Stratégie : médiane départementale, puis médiane nationale en fallback.

In [9]:
print("Imputation des valeurs manquantes...")

numeric_cols = [
    'nb_naissances', 'nb_deces', 'solde_naturel',
    'mediane_revenu_disp', 'gini', 'taux_pauvrete',
    'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct',
    'pct_bac_plus', 'pct_sans_diplome',
]

# S'assurer que code_dep est disponible
if 'code_dep' not in df_gold.columns or df_gold['code_dep'].isna().all():
    df_gold['code_dep'] = df_gold['code_commune'].str[:2]
    df_gold.loc[df_gold['code_dep'] == '2A', 'code_dep'] = '2A'
    df_gold.loc[df_gold['code_dep'] == '2B', 'code_dep'] = '2B'

for col in numeric_cols:
    if col not in df_gold.columns:
        df_gold[col] = np.nan
        continue
    
    n_missing_before = df_gold[col].isna().sum()
    if n_missing_before == 0:
        continue
    
    # Médiane départementale
    dep_median = df_gold.groupby('code_dep')[col].transform('median')
    df_gold[col] = df_gold[col].fillna(dep_median)
    
    # Médiane nationale en fallback
    nat_median = df_gold[col].median()
    df_gold[col] = df_gold[col].fillna(nat_median)
    
    n_missing_after = df_gold[col].isna().sum()
    if n_missing_before > 0:
        print(f"  {col}: {n_missing_before:,} → {n_missing_after:,} manquants")

print("\nTaux de remplissage après imputation (%) :")
fill_rate_after = (df_gold.notna().sum() / len(df_gold) * 100).round(1)
print(fill_rate_after[fill_rate_after < 100].to_string() or "  Toutes colonnes à 100%")

Imputation des valeurs manquantes...
  nb_naissances: 335 → 0 manquants
  nb_deces: 335 → 0 manquants
  solde_naturel: 335 → 0 manquants
  mediane_revenu_disp: 17,997 → 0 manquants
  gini: 147,700 → 0 manquants
  taux_pauvrete: 173,925 → 173,925 manquants
  cadres_pct: 1,114 → 0 manquants
  ouvriers_pct: 1,114 → 0 manquants
  employes_pct: 1,114 → 0 manquants
  artisans_pct: 1,114 → 0 manquants
  pct_bac_plus: 340 → 0 manquants
  pct_sans_diplome: 340 → 0 manquants

Taux de remplissage après imputation (%) :
taux_pauvrete    0.0


/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/usr/local

## 7. Insertion dans gold_france.features_communes

In [10]:
# Sélectionner et ordonner les colonnes Gold
gold_cols = [
    'code_commune', 'libelle', 'code_dep', 'annee',
    'pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite', 'pct_divers',
    'bloc_dominant',
    'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct',
    'pct_bac_plus', 'pct_sans_diplome',
    'nb_naissances', 'nb_deces', 'solde_naturel',
    'typologie_territoire',
]

# Ajouter revenus si disponibles
for col in ['mediane_revenu_disp', 'gini', 'taux_pauvrete']:
    if col in df_gold.columns:
        gold_cols.append(col)

df_insert = df_gold[[c for c in gold_cols if c in df_gold.columns]].copy()

# Colonnes manquantes → NaN
for c in gold_cols:
    if c not in df_insert.columns:
        df_insert[c] = np.nan
        print(f"  WARN: colonne {c} absente, mise à NaN")

# Types
df_insert['annee'] = df_insert['annee'].astype('Int16')
for col in ['nb_naissances', 'nb_deces', 'solde_naturel']:
    if col in df_insert.columns:
        df_insert[col] = pd.to_numeric(df_insert[col], errors='coerce').astype('Int64')

print(f"Lignes à insérer : {len(df_insert):,}")
print(f"Colonnes : {df_insert.columns.tolist()}")
df_insert.head(3)

Lignes à insérer : 173,925
Colonnes : ['code_commune', 'libelle', 'code_dep', 'annee', 'pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite', 'pct_divers', 'bloc_dominant', 'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct', 'pct_bac_plus', 'pct_sans_diplome', 'nb_naissances', 'nb_deces', 'solde_naturel', 'typologie_territoire', 'mediane_revenu_disp', 'gini', 'taux_pauvrete']


,code_commune,libelle,code_dep,annee,pct_gauche,pct_centre,pct_droite,pct_extremedroite,pct_divers,bloc_dominant,...,artisans_pct,pct_bac_plus,pct_sans_diplome,nb_naissances,nb_deces,solde_naturel,typologie_territoire,mediane_revenu_disp,gini,taux_pauvrete
0,01001,ABERGEMENT CLEMENCIAT,01,2002,28.805621,14.285714,32.318501,24.590164,0.0,Droite,...,4.762,29.601,15.491,8,4,4,rural,25820.0,0.242,NaN
1,01001,ABERGEMENT CLEMENCIAT,01,2007,24.904215,19.731801,43.486590,11.877395,0.0,Droite,...,4.762,29.601,15.491,8,4,4,rural,25820.0,0.242,NaN
2,01001,ABERGEMENT CLEMENCIAT,01,2012,30.861723,10.821643,33.066132,25.250501,0.0,Droite,...,4.762,29.601,15.491,5,7,-2,rural,25820.0,0.242,NaN


In [11]:
import time

print("Insertion dans gold_france.features_communes...")
t_insert = time.time()

# Truncate avant insertion
with engine.begin() as conn:
    conn.execute(text("TRUNCATE gold_france.features_communes"))

# Insérer par chunks
CHUNK = 5000
total = len(df_insert)
for i in range(0, total, CHUNK):
    df_insert.iloc[i:i+CHUNK].to_sql(
        'features_communes',
        engine,
        schema='gold_france',
        if_exists='append',
        index=False,
        method='multi',
        chunksize=CHUNK
    )
    if (i // CHUNK + 1) % 20 == 0:
        pct = min(100, (i+CHUNK)/total*100)
        print(f"  {i+CHUNK:,}/{total:,} lignes ({pct:.0f}%) | {time.time()-t_insert:.0f}s")

elapsed = time.time() - t_insert

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM gold_france.features_communes")).scalar()

print(f"\ngold_france.features_communes : {n:,} lignes insérées en {elapsed:.1f}s")

Insertion dans gold_france.features_communes...


  100,000/173,925 lignes (57%) | 13s



gold_france.features_communes : 173,925 lignes insérées en 22.7s


## 8. Validation Gold France

In [12]:
df_val = pd.read_sql("""
    SELECT annee,
           COUNT(*) AS nb_communes,
           ROUND(AVG(pct_gauche)::numeric, 1)         AS moy_gauche,
           ROUND(AVG(pct_centre)::numeric, 1)         AS moy_centre,
           ROUND(AVG(pct_droite)::numeric, 1)         AS moy_droite,
           ROUND(AVG(pct_extremedroite)::numeric, 1)  AS moy_xd,
           ROUND(AVG(cadres_pct)::numeric, 1)         AS moy_cadres_pct,
           ROUND(AVG(pct_bac_plus)::numeric, 1)       AS moy_bac_plus
    FROM gold_france.features_communes
    GROUP BY annee
    ORDER BY annee
""", engine)

print("=" * 80)
print("VALIDATION GOLD FRANCE - gold_france.features_communes")
print("=" * 80)
print(df_val.to_string(index=False))

# Distribution typologies
df_typo = pd.read_sql("""
    SELECT typologie_territoire, COUNT(DISTINCT code_commune) AS nb_communes
    FROM gold_france.features_communes
    GROUP BY typologie_territoire
    ORDER BY nb_communes DESC
""", engine)

print("\nDistribution typologies territoire :")
print(df_typo.to_string(index=False))

# Distribution blocs 2022
df_blocs_2022 = pd.read_sql("""
    SELECT bloc_dominant, COUNT(*) AS nb_communes
    FROM gold_france.features_communes
    WHERE annee = 2022
    GROUP BY bloc_dominant
    ORDER BY nb_communes DESC
""", engine)

print("\nBlocs dominants 2022 :")
print(df_blocs_2022.to_string(index=False))

print("\nFeature engineering Gold France terminé !")

VALIDATION GOLD FRANCE - gold_france.features_communes
 annee  nb_communes  moy_gauche  moy_centre  moy_droite  moy_xd  moy_cadres_pct  moy_bac_plus
  2002        34777        28.3        11.2        38.8    21.7            12.3          27.0
  2007        34780        31.6        18.3        37.5    12.6            12.3          27.0
  2012        34794        39.5         9.5        29.6    21.4            12.3          27.0
  2017        34791        24.5        22.4        26.5    26.4            12.3          27.0
  2022        34783        25.7        30.4         7.7    36.1            12.3          27.0

Distribution typologies territoire :
typologie_territoire  nb_communes
               rural        29538
          periurbain         4310
              urbain          952

Blocs dominants 2022 :
bloc_dominant  nb_communes
ExtremeDroite        20193
       Centre         9485
       Gauche         5079
       Droite           26

Feature engineering Gold France terminé !
